In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np
import folium
import requests
import branca.colormap as cm
from folium.plugins import DualMap
from shapely.geometry import Polygon, MultiPolygon, Point, box
from shapely.ops import voronoi_diagram

df = pd.read_csv('dados/processados/votacao_geolocalizada.csv', sep=';')
df = df.dropna(subset=['NR_LATITUDE', 'NR_LONGITUDE']).copy()

In [3]:
print("Colunas disponíveis:", df.columns.tolist())
display(df.head())

Colunas disponíveis: ['CD_MUNICIPIO', 'NR_ZONA', 'NR_SECAO', 'NR_TURNO', 'NR_VOTAVEL', 'NM_VOTAVEL', 'QT_VOTOS', 'NR_LOCAL_VOTACAO', 'NM_LOCAL_VOTACAO', 'DS_ENDERECO', 'NM_BAIRRO', 'NR_LATITUDE', 'NR_LONGITUDE']


,CD_MUNICIPIO,NR_ZONA,NR_SECAO,NR_TURNO,NR_VOTAVEL,NM_VOTAVEL,QT_VOTOS,NR_LOCAL_VOTACAO,NM_LOCAL_VOTACAO,DS_ENDERECO,NM_BAIRRO,NR_LATITUDE,NR_LONGITUDE
0,53430,263,100,1,13,LUIZ INÁCIO LULA DA SILVA,86,1287,ESCOLA ESTADUAL DR. AFONSO VIANA,"RUA FREI EUSTAQUIO, 382",NOSSA SENHORA DAS GRAÇAS,-19.446732,-44.24742
1,53430,263,100,1,14,RENAN ANTONIO FERREIRA DOS SANTOS,5,1287,ESCOLA ESTADUAL DR. AFONSO VIANA,"RUA FREI EUSTAQUIO, 382",NOSSA SENHORA DAS GRAÇAS,-19.446732,-44.24742
2,53430,263,100,1,22,FLAVIO NANTES BOLSONARO,127,1287,ESCOLA ESTADUAL DR. AFONSO VIANA,"RUA FREI EUSTAQUIO, 382",NOSSA SENHORA DAS GRAÇAS,-19.446732,-44.24742
3,53430,263,100,1,30,ROMEU ZEMA NETO,7,1287,ESCOLA ESTADUAL DR. AFONSO VIANA,"RUA FREI EUSTAQUIO, 382",NOSSA SENHORA DAS GRAÇAS,-19.446732,-44.24742
4,53430,263,100,1,55,RONALDO RAMOS CAIADO,4,1287,ESCOLA ESTADUAL DR. AFONSO VIANA,"RUA FREI EUSTAQUIO, 382",NOSSA SENHORA DAS GRAÇAS,-19.446732,-44.24742


### Preparação dos Dados Geográficos (Sete Lagoas - MG)

Como a base do TSE traz apenas a latitude e a longitude exatas de cada local de votação (pontos), precisamos transformar essas coordenadas em áreas (polígonos) para conseguir montar os mapas:

1. **Contorno do Município:** Baixamos a malha oficial de Sete Lagoas direto da API do IBGE (código `3167202`) para ter o limite exato da cidade.
2. **Agrupamento dos Locais:** Removemos coordenadas duplicadas para identificar cada escola/local de votação de forma única (`id_local`), evitando que os pontos ficassem sobrepostos.
3. **Criação das Regiões (Voronoi):** Usamos o diagrama de Voronoi (`shapely`) para dividir o mapa em áreas ao redor de cada local de votação e recortamos o resultado usando o contorno oficial de Sete Lagoas, garantindo que as regiões fiquem dentro dos limites reais do município.
4. **Foco e Visualização:** Limitamos o zoom do mapa apenas à região de Sete Lagoas (`bounds_sl`) e aplicamos um ajuste de estilo (CSS) para aumentar o tamanho das legendas e criar uma linha divisória entre os dois mapas lado a lado.

In [4]:
geojson_cidade = None
poligono_cidade = None
try:
    url_sl = "https://servicodados.ibge.gov.br/api/v3/malhas/municipios/3167202?formato=application/vnd.geo+json"
    resp = requests.get(url_sl, headers={"User-Agent": "Mozilla/5.0"}, timeout=15)
    if resp.status_code == 200:
        geojson_cidade = resp.json()
        gdf_mun = gpd.read_file(resp.text).to_crs(epsg=4326)
        poligono_cidade = gdf_mun.geometry.union_all()
except Exception:
    pass

# Agrupar locais únicos de votação pelas coordenadas originais
locais = (
    df[['NR_ZONA', 'NM_BAIRRO', 'NM_LOCAL_VOTACAO', 'NR_LATITUDE', 'NR_LONGITUDE']]
    .drop_duplicates(subset=['NR_LATITUDE', 'NR_LONGITUDE'])
    .reset_index(drop=True)
)
locais['id_local'] = locais.index.astype(str)
df = df.merge(locais[['NR_LATITUDE', 'NR_LONGITUDE', 'id_local']], on=['NR_LATITUDE', 'NR_LONGITUDE'], how='left')

# Gerar polígonos de influência (Voronoi) recortados pelo limite real de Sete Lagoas
pontos = gpd.GeoSeries([Point(xy) for xy in zip(locais['NR_LONGITUDE'], locais['NR_LATITUDE'])], crs="EPSG:4326")
minx, miny, maxx, maxy = pontos.total_bounds
envelope = poligono_cidade.buffer(0.05) if poligono_cidade is not None else box(minx - 0.02, miny - 0.02, maxx + 0.02, maxy + 0.02)
recorte_final = poligono_cidade if poligono_cidade is not None else envelope

regioes_voronoi = voronoi_diagram(pontos.union_all(), envelope=envelope)

poligonos = []
for _, row in locais.iterrows():
    ponto = Point(row['NR_LONGITUDE'], row['NR_LATITUDE'])
    poly = next((g.intersection(recorte_final) for g in regioes_voronoi.geoms if g.contains(ponto) or g.touches(ponto)), None)
    poligonos.append(poly)

gdf_locais = gpd.GeoDataFrame(locais, geometry=poligonos, crs="EPSG:4326")
gdf_locais = gdf_locais[gdf_locais.geometry.notnull()].copy()

# Limites para focar apenas em Sete Lagoas
bounds_sl = [[miny - 0.015, minx - 0.015], [maxy + 0.015, maxx + 0.015]]

# CSS para legendas grandes e divisória central nítida
css_customizado = """
<style>
.legend {
    background-color: rgba(255, 255, 255, 0.96) !important;
    padding: 12px 16px !important;
    border: 2px solid #2c3e50 !important;
    border-radius: 8px !important;
    box-shadow: 0 4px 10px rgba(0,0,0,0.25) !important;
}
.legend svg {
    width: 340px !important;
    height: 65px !important;
}
.legend text {
    font-size: 14px !important;
    font-weight: bold !important;
    fill: #1e272e !important;
}
.leaflet-container:first-of-type {
    border-right: 5px solid #2c3e50 !important;
    box-sizing: border-box !important;
}
</style>
"""

print(f"Áreas geradas com sucesso: {len(gdf_locais)} regiões eleitorais dentro de Sete Lagoas.")

Áreas geradas com sucesso: 78 regiões eleitorais dentro de Sete Lagoas.


### Leitura 1ª e 2ª turno

In [5]:
def resumir_vitorias(df_turno, numero_turno):
    # Filtra apenas votos válidos
    df_val = df_turno[~df_turno['NM_VOTAVEL'].str.upper().isin(['VOTO BRANCO', 'VOTO NULO'])].copy()
    
    # Total geral na cidade
    total_geral = df_val.groupby('NM_VOTAVEL')['QT_VOTOS'].sum().nlargest(2)
    perc_geral = (total_geral / df_val['QT_VOTOS'].sum() * 100).round(2)
    
    print(f"================ {numero_turno}º TURNO (SETE LAGOAS) ================")
    for cand in total_geral.index:
        print(f"Total {cand}: {total_geral[cand]} votos ({perc_geral[cand]}%)")
    print("-" * 55)
    
    # Agrupa por local de votação
    chaves = ['NR_ZONA', 'NM_BAIRRO', 'NM_LOCAL_VOTACAO']
    totais_local = df_val.groupby(chaves)['QT_VOTOS'].sum().reset_index(name='total_local')
    votos_local = df_val.groupby(chaves + ['NM_VOTAVEL'])['QT_VOTOS'].sum().reset_index().merge(totais_local, on=chaves)
    votos_local['percentual'] = (votos_local['QT_VOTOS'] / votos_local['total_local'] * 100).round(2)
    
    # Filtra o vencedor de cada local
    vencedores = votos_local.sort_values('QT_VOTOS', ascending=False).drop_duplicates(chaves)
    
    for cand in total_geral.index:
        vitorias = vencedores[vencedores['NM_VOTAVEL'] == cand].sort_values('percentual', ascending=False)
        bairros_unicos = vitorias['NM_BAIRRO'].unique().tolist()
        
        print(f"\n>>> ONDE {cand} GANHOU ({len(vitorias)} locais de votação):")
        print(f"Bairros: {', '.join(bairros_unicos)}")
        print("Top 5 maiores percentuais:")
        display(vitorias[['NR_ZONA', 'NM_BAIRRO', 'NM_LOCAL_VOTACAO', 'QT_VOTOS', 'percentual']].head(5))

# Rodar para o 1º e 2º turnos
resumir_vitorias(df[df['NR_TURNO'] == 1], 1)
resumir_vitorias(df[df['NR_TURNO'] == 2], 2)

================ 1º TURNO (SETE LAGOAS) ================
Total FLAVIO NANTES BOLSONARO: 70304 votos (54.89%)
Total LUIZ INÁCIO LULA DA SILVA: 44239 votos (34.54%)
-------------------------------------------------------

>>> ONDE FLAVIO NANTES BOLSONARO GANHOU (77 locais de votação):
Bairros: CENTRO, BARREIRO DE BAIXO, BOA VISTA, ELDORADO, ESTIVA, VALE DAS PALMEIRAS, MORRO DO CLARO, SÃO GERALDO, RIACHO DO CAMPO, CANAAN, NOSSA SENHORA DO CARMO , MORRO REDONDO, CHÁCARA DO PAIVA, JARDIM CAMBUI, SANTA LUZIA, BARREIRO DE CIMA, INTERLAGOS, AEROPORTO INDUSTRIAL, VÁRZEA, SANTO ANTÔNIO, CATARINA, LONTRA, SANTA ROSA, FAZENDA VELHA, NOVO HORIZONTE, BOM JARDIM, NOVA CIDADE, MANOA, QUINTAS DA VARGINHA, MONTREAL, SILVA XAVIER, JK, OLINTO ALVIM, INDÚSTRIAS, JARDIM EUROPA, JARDIM PRIMAVERA II, INDUSTRIAL, PADRE TEODORO, LUXEMBURGO, VAPABUÇU, BERNARDO VALADARES, CDI, ESPERANÇA, OROZIMBO MACEDO, FÁTIMA, CIDADE DE DEUS, BELO VALE, CANADÁ, EMÍLIA, JARDIM ARIZONA, NOSSA SENHORA DAS GRAÇAS, ALVORADA, TAMANDU

,NR_ZONA,NM_BAIRRO,NM_LOCAL_VOTACAO,QT_VOTOS,percentual
41,263,CENTRO,FACULDADES PROMOVE,209,68.75
429,322,BARREIRO DE BAIXO,ESCOLA MUNICIPAL ALFREDO PEREIRA DE SOUZA,245,66.76
26,263,CENTRO,ESCOLA ESTADUAL DR. ULISSES VASCONCELOS,418,64.51
453,322,BOA VISTA,ESCOLA ESTADUAL PROFESSOR JOÃO FERNANDINO JÚNIOR,367,64.05
48,263,CENTRO,UNIVERSIDADE UNOPAR,1641,63.90



>>> ONDE LUIZ INÁCIO LULA DA SILVA GANHOU (1 locais de votação):
Bairros: DONA SILVIA
Top 5 maiores percentuais:


,NR_ZONA,NM_BAIRRO,NM_LOCAL_VOTACAO,QT_VOTOS,percentual
274,264,DONA SILVIA,ESCOLA MUNICIPAL JOSÉ JACINTO MARTINS DE GODOY...,297,52.66


================ 2º TURNO (SETE LAGOAS) ================
Total FLAVIO NANTES BOLSONARO: 76327 votos (56.67%)
Total LUIZ INÁCIO LULA DA SILVA: 58368 votos (43.33%)
-------------------------------------------------------

>>> ONDE FLAVIO NANTES BOLSONARO GANHOU (71 locais de votação):
Bairros: CENTRO, BARREIRO DE BAIXO, BOA VISTA, ELDORADO, VALE DAS PALMEIRAS, MORRO DO CLARO, CHÁCARA DO PAIVA, JARDIM CAMBUI, CANAAN, SÃO GERALDO, ESTIVA, NOSSA SENHORA DO CARMO , SANTA LUZIA, FAZENDA VELHA, INDUSTRIAL, OLINTO ALVIM, SILVA XAVIER, MORRO REDONDO, AEROPORTO INDUSTRIAL, BARREIRO DE CIMA, SANTO ANTÔNIO, CATARINA, BOM JARDIM, JARDIM EUROPA, SANTA ROSA, JARDIM PRIMAVERA II, NOVA CIDADE, VÁRZEA, RIACHO DO CAMPO, JK, LONTRA, PADRE TEODORO, NOVO HORIZONTE, VAPABUÇU, INDÚSTRIAS, INTERLAGOS, ESPERANÇA, MONTREAL, MANOA, BERNARDO VALADARES, LUXEMBURGO, CDI, JARDIM ARIZONA, QUINTAS DA VARGINHA, FÁTIMA, NOSSA SENHORA DAS GRAÇAS, EMÍLIA, CANADÁ, BELO VALE, BOUGANVILLE I, TAMANDUÁ, CIDADE DE DEUS, OROZIMBO 

,NR_ZONA,NM_BAIRRO,NM_LOCAL_VOTACAO,QT_VOTOS,percentual
10,263,CENTRO,FACULDADES PROMOVE,220,69.40
100,322,BARREIRO DE BAIXO,ESCOLA MUNICIPAL ALFREDO PEREIRA DE SOUZA,270,68.35
6,263,CENTRO,ESCOLA ESTADUAL DR. ULISSES VASCONCELOS,449,67.62
106,322,BOA VISTA,ESCOLA ESTADUAL PROFESSOR JOÃO FERNANDINO JÚNIOR,387,66.72
12,263,CENTRO,UNIVERSIDADE UNOPAR,1744,66.46



>>> ONDE LUIZ INÁCIO LULA DA SILVA GANHOU (7 locais de votação):
Bairros: DONA SILVIA, SANTA FELICIDADE, JARDIM DOS PEQUIS, ITAPOÃ II, CIDADE DE DEUS, VERDE VALE, LONTRINHA
Top 5 maiores percentuais:


,NR_ZONA,NM_BAIRRO,NM_LOCAL_VOTACAO,QT_VOTOS,percentual
65,264,DONA SILVIA,ESCOLA MUNICIPAL JOSÉ JACINTO MARTINS DE GODOY...,380,62.91
97,264,SANTA FELICIDADE,ESCOLA MUNICIPAL MARILZA FLEURY COSTA FIGUEIREDO,298,55.19
79,264,JARDIM DOS PEQUIS,ESCOLA MUNICIPAL PROFESSOR RAIMUNDO GRAVITO,503,54.61
125,322,ITAPOÃ II,E. M. DOUTOR ENÍZIO ANTÔNIO VIANA,1230,50.81
61,264,CIDADE DE DEUS,ESCOLA ESTADUAL RUTH BRANDÃO DE AZEREDO,578,50.39


### Construção dos Mapas Comparativos

Nesta etapa, criamos a função responsável por montar a visualização comparativa entre os dois principais candidatos (Lula e Bolsonaro) tanto para o 1º quanto para o 2º turno:

1. **Cálculo de Percentuais e Vencedor:** Somamos os votos válidos de cada área de votação (`id_local`), calculamos a porcentagem de cada candidato e identificamos quem ficou em primeiro lugar em cada região.
2. **Mapas Sincronizados (`DualMap`):** Usamos o plugin `DualMap` do Folium para colocar os dois mapas lado a lado sobre o mapa base `OpenStreetMap`, travando o enquadramento em Sete Lagoas. Assim, ao dar zoom ou arrastar um lado, o outro acompanha na mesma posição.
3. **Coloração por Dominância:** No mapa da esquerda (vermelho), colorimos apenas as regiões onde o Lula venceu; no da direita (verde), apenas onde o Bolsonaro venceu. Quanto mais forte a cor, maior a porcentagem de votos do candidato naquela área, enquanto as regiões onde ele perdeu ficam em cinza claro para não atrapalhar a leitura.
4. **Interatividade e Exportação:** Adicionamos controle de camadas, legendas ampliadas e tooltips personalizados em HTML/CSS que mostram a zona eleitoral, o bairro, o local de votação, o candidato líder e o placar exato ao passar o mouse. Por fim, cada mapa é salvo em formato `.html`, dentro da pasta `mapas`, para visualização no navegador.

In [6]:
def gerar_mapa_dominancia_lado_a_lado(df_turno, numero_turno, nome_arquivo_html):
    # Calcular totais e percentuais por área
    totais = df_turno.groupby('id_local')['QT_VOTOS'].sum().reset_index(name='total_votos')
    votos = df_turno.groupby(['id_local', 'NM_VOTAVEL'])['QT_VOTOS'].sum().reset_index().merge(totais, on='id_local')
    votos['percentual'] = (votos['QT_VOTOS'] / votos['total_votos'] * 100).round(2)
    
    # Identificar o candidato líder (vencedor) em cada área
    vencedores = votos.sort_values('QT_VOTOS', ascending=False).drop_duplicates('id_local')[['id_local', 'NM_VOTAVEL', 'percentual']]
    vencedores = vencedores.rename(columns={'NM_VOTAVEL': 'vencedor', 'percentual': 'perc_vencedor'})
    
    # Selecionar os 2 principais candidatos e ordenar (Lula à esquerda, Bolsonaro à direita)
    top2 = df_turno.groupby('NM_VOTAVEL')['QT_VOTOS'].sum().nlargest(2).index.tolist()
    top2_ordenados = sorted(top2, key=lambda x: 0 if 'LULA' in x.upper() else 1)
    cand_esq, cand_dir = top2_ordenados[0], top2_ordenados[1]
    
    # Pivotar percentuais para exibir o placar completo nos tooltips
    pivot = votos.pivot(index='id_local', columns='NM_VOTAVEL', values='percentual').fillna(0).reset_index()
    pivot = pivot.merge(vencedores, on='id_local', how='left')
    
    gdf_mapa = gdf_locais.merge(pivot, on='id_local', how='left')
    gdf_mapa['lider_texto'] = gdf_mapa['vencedor'] + ' (' + gdf_mapa['perc_vencedor'].astype(str) + '%)'
    gdf_mapa['placar_texto'] = (
        cand_esq.split()[0] + ': ' + gdf_mapa[cand_esq].astype(str) + '% | ' +
        cand_dir.split()[0] + ': ' + gdf_mapa[cand_dir].astype(str) + '%'
    )
    
    # Criar DualMap focado exclusivamente em Sete Lagoas
    mapa_duplo = DualMap(
        location=[-19.4658, -44.2467],
        zoom_start=13,
        min_zoom=12,
        max_bounds=True,
        min_lat=bounds_sl[0][0] - 0.05,
        max_lat=bounds_sl[1][0] + 0.05,
        min_lon=bounds_sl[0][1] - 0.05,
        max_lon=bounds_sl[1][1] + 0.05,
        tiles='OpenStreetMap'
    )
    mapa_duplo.m1.fit_bounds(bounds_sl)
    mapa_duplo.m2.fit_bounds(bounds_sl)
    mapa_duplo.get_root().html.add_child(folium.Element(css_customizado))
    
    lados = [
        (mapa_duplo.m1, cand_esq, cm.linear.Reds_09, '#7f0000'),
        (mapa_duplo.m2, cand_dir, cm.linear.Greens_09, '#00441b')
    ]
    
    for mapa_lado, cand, paleta_base, cor_destaque in lados:
        # Escala calibrada nas áreas onde este candidato venceu
        vitorias_cand = gdf_mapa[gdf_mapa['vencedor'] == cand][cand]
        vmin = vitorias_cand.min() if len(vitorias_cand) > 0 else 30
        vmax = vitorias_cand.max() if len(vitorias_cand) > 0 else 70
        if vmin == vmax:
            vmin = max(0, vmin - 10)
            
        colormap = paleta_base.scale(vmin, vmax)
        colormap.caption = f'{numero_turno}º Turno: Áreas Vencidas por {cand.split()[0]} (%)'
        
        # Função de estilo: colore com intensidade onde o candidato ganhou; cinza neutro onde perdeu
        def criar_estilo(candidato_atual, cmap):
            def style_fn(feature):
                props = feature['properties']
                if props.get('vencedor') == candidato_atual:
                    val = props.get(candidato_atual, 0)
                    return {
                        'fillColor': cmap(val),
                        'color': '#2c3e50',
                        'weight': 1.5,
                        'fillOpacity': 0.72
                    }
                else:
                    return {
                        'fillColor': '#dfe6e9',
                        'color': '#636e72',
                        'weight': 1,
                        'fillOpacity': 0.25
                    }
            return style_fn
        
        tooltip = folium.GeoJsonTooltip(
            fields=['NR_ZONA', 'NM_BAIRRO', 'NM_LOCAL_VOTACAO', 'lider_texto', 'placar_texto'],
            aliases=['Zona Eleitoral:', 'Bairro:', 'Local de Votação:', f'Líder ({numero_turno}º Turno):', 'Placar Relativo:'],
            localize=True,
            labels=True,
            style=f"""
                background-color: #ffffff;
                border: 2px solid {cor_destaque};
                border-left: 6px solid {cor_destaque};
                border-radius: 6px;
                box-shadow: 3px 3px 8px rgba(0,0,0,0.25);
                font-family: Arial, sans-serif;
                font-size: 13px;
                padding: 8px;
            """
        )
        
        folium.GeoJson(
            gdf_mapa,
            name=f"Zonas Vencidas - {cand.split()[0]}",
            style_function=criar_estilo(cand, colormap),
            tooltip=tooltip
        ).add_to(mapa_lado)
        
        colormap.add_to(mapa_lado)
        folium.LayerControl().add_to(mapa_lado)
        
    mapa_duplo.save(nome_arquivo_html)
    print(f"Mapa salvo com sucesso: {nome_arquivo_html}")
    return mapa_duplo

### Geração dos Mapas e Análise dos Resultados (1º e 2º Turnos)

Neste bloco final, filtramos o dataset por turno (`NR_TURNO == 1` e `NR_TURNO == 2`), removemos os votos brancos e nulos para trabalhar apenas com os votos válidos e geramos os dois mapas comparativos (`mapa_1turno_comparativo.html` e `mapa_2turno_comparativo.html`).

### Relatório de Análise 

#### 1. Comparação do 1º Turno (`mapa_1turno_comparativo.html`)
* **O fato:** No 1º turno em Sete Lagoas, **Flávio Bolsonaro** teve ampla vantagem no município, somando **70.304 votos (54,89%)** contra **44.239 votos (34,54%)** de **Lula**. No mapa, isso fica claro pela predominância quase total da cor verde na maior parte da cidade: Flávio venceu em **77 dos 78 locais de votação**, com destaque para a região central e bairros tradicionais como **Centro** (chegando a 68,75% nas Faculdades Promove e 64,51% na E.E. Dr. Ulisses Vasconcelos), **Barreiro de Baixo** (66,76%) e **Boa Vista** (64,05%). Já **Lula** venceu em apenas **1 local de votação** no 1º turno, situado no bairro **Dona Sílvia** (E.M. José Jacinto Martins de Godoy, com 52,66%).
* **O design:** Optamos por colocar os mapas lado a lado (`DualMap`) com escalas sequenciais ligadas à identidade visual de cada candidato (`Reds` para Lula e `Greens` para Flávio Bolsonaro). Colorimos apenas as áreas vencidas por cada um com opacidade de `0.72` (deixando o restante em cinza bem transparente), o que evita poluição visual, mantém os nomes das ruas legíveis e facilita comparar onde cada candidato dominou.

#### 2. Comparação do 2º Turno (`mapa_2turno_comparativo.html`)
* **O fato:** No 2º turno, **Flávio Bolsonaro** manteve a liderança geral no município com **76.327 votos (56,67%)** em **71 locais de votação**, consolidando ainda mais seus redutos centrais — como o **Centro** (69,40% nas Faculdades Promove), **Barreiro de Baixo** (68,35%) e **Boa Vista** (66,72%). Por outro lado, **Lula** apresentou um crescimento proporcional maior em relação ao 1º turno, subindo para **58.368 votos (43,33%)** e ampliando sua área de liderança de 1 para **7 locais de votação**. Além de fortalecer sua vitória no bairro **Dona Sílvia** (subindo de 52,66% para 62,91%), ele virou o resultado e passou a liderar em bairros mais periféricos e populosos, como **Santa Felicidade** (55,19%), **Jardim dos Pequis** (54,61%), **Itapoã II** (50,81%), **Cidade de Deus** (50,39%), **Verde Vale** e **Lontrinha**.
* **O design:** Mantivemos a mesma estrutura sincronizada lado a lado com legenda ampliada e tooltips customizados em HTML/CSS exibindo a zona eleitoral, o bairro, o local de votação e o placar exato de cada seção. A padronização das cores e da escala entre os dois turnos permite enxergar rapidamente como a disputa se polarizou entre o centro/bairros consolidados (verde mais intenso) e as regiões periféricas onde o vermelho ganhou força no 2º turno.

In [7]:
# 1º Turno
df_t1 = df[(df['NR_TURNO'] == 1) & (~df['NM_VOTAVEL'].str.upper().isin(['VOTO BRANCO', 'VOTO NULO']))].copy()
mapa_1t = gerar_mapa_dominancia_lado_a_lado(df_t1, numero_turno=1, nome_arquivo_html="mapas/mapa_1turno_comparativo.html")
display(mapa_1t)

# 2º Turno
df_t2 = df[(df['NR_TURNO'] == 2) & (~df['NM_VOTAVEL'].str.upper().isin(['VOTO BRANCO', 'VOTO NULO']))].copy()
mapa_2t = gerar_mapa_dominancia_lado_a_lado(df_t2, numero_turno=2, nome_arquivo_html="mapas/mapa_2turno_comparativo.html")
display(mapa_2t)

Mapa salvo com sucesso: mapas/mapa_1turno_comparativo.html


Mapa salvo com sucesso: mapas/mapa_2turno_comparativo.html
